# 🕵️ Dia 54 — Forense de memória: conexões de rede e código injetado

**Investigador Cyber 90 Dias** · Fase 4 · Rank Jutsu `B`

> Baseado no *Day 54* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-4-digital-forensics-incident-investigation/day-54.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

O `msa-atualizador` tem duas conexões com 198.51.100.140, cinco minutos de diferença. *"É um beacon!"*, você diz. Rafa: *"Duas linhas. O que cada uma mostra, separadamente?"*

> ⚠️ Sem imagem de memória real (nem no original, nem no caso BR). `netscan` e `malfind` são texto sintético; os bytes do *hexdump* são ilustrativos e **nada é executável**.

Neste notebook você vai:

1. **Filtrar o netscan** (o `awk` do passo 1) e classificar os remotos.
2. **Julgar cada região do malfind** com o fluxograma do conceito, como código.
3. **Testar se "cinco minutos" é padrão** quando uma série independente entra.
4. **Gerar o grafo processo-rede.**
5. **Jogar o quiz do "o que esta linha prova".**


In [ ]:
# ------------------------------------------------------------
# RECEBER NETSCAN E MALFIND DO WS-FIN-07 (EVID-003)
# Plano A: baixar do repositório original (GitHub raw), com urllib.
# Plano B: usar a cópia embutida logo abaixo (o arquivo inteiro,
#          copiado do upstream/), para o notebook rodar sem internet.
# ------------------------------------------------------------
import urllib.request      # baixa arquivos da web, sem instalar nada
import hashlib             # SHA-256: a impressão digital dos bytes

FORCAR_OFFLINE = False     # True = nem tenta a rede (útil em sala sem Wi-Fi)
URL_BASE = 'https://raw.githubusercontent.com/thiefdroppers/cyber-investigator-skills/main/curriculum/phase-4-digital-forensics-incident-investigation/resources/'

RESERVA = {}             # caminho do arquivo -> conteúdo embutido (plano B)
RESERVA['case-lab-p4/memory/windows.netscan.txt'] = ''.join((
    '# SYNTHETIC TRAINING DATA (case LAB-P4). Hand-written to resemble Volatility 3 windows.netscan\n'
    '# output for fictional host WS-FIN-07 (10.10.40.57). Public IPs are RFC 5737 documentation\n'
    '# addresses. Tab-separated.\n'
    'Offset\tProto\tLocalAddr\tLocalPort\tForeignAddr\tForeignPort\tState\tPID\tOwner\tCreated\n'
    '0xa3066fb3a010\tTCPv4\t0.0.0.0\t135\t0.0.0.0\t0\tLISTENING\t852\tsvchost.exe\t2026-03-13 12:58:04.000000 UTC\n'
    '0xa3066fb3a9e0\tTCPv4\t0.0.0.0\t445\t0.0.0.0\t0\tLISTENING\t4\tSystem\t2026-03-13 12:58:06.000000 UTC\n'
    '0xa3066fc10c20\tTCPv4\t0.0.0.0\t49664\t0.0.0.0\t0\tLISTENING\t672\tlsass.exe\t2026-03-13 12:58:04.000000 UTC\n'
    '0xa30671c90a60\tTCPv4\t10.10.40.57\t50344\t192.0.2.30\t443\tESTABLISHED\t6632\tOUTLOOK.EXE\t2026-03-14 04:58:12.000000 UTC\n'
    '0xa30671d99b20\tTCPv4\t10.10.40.57\t50402\t192.0.2.31\t443\tESTABLISHED\t6208\tmsedge.exe\t2026-03-14 05:02:40.000000 UTC\n'
    '0xa30672e9d8a0\tTCPv4\t10.10.40.57\t50811\t198.51.100.23\t443\tESTABLISHED\t7488\tsynchelper.exe\t2026-03-14 05:07:31.000000 UTC\n'
    '0xa30672e9f010\tTCPv4\t10.10.40.57\t50799\t198.51.100.23\t443\tCLOSED\t7488\tsynchelper.exe\t2026-03-14 05:02:30.000000 UTC\n'
    '0xa30672f55d30\tTCPv4\t10.10.40.57\t50822\t203.0.113.80\t8443\tSYN_SENT\t5124\tsvchost.exe\t2026-03-14 05:09:58.000000 UTC\n'
    '0xa30672f60a10\tUDPv4\t10.10.40.57\t55122\t*\t0\t\t7488\tsynchelper.exe\t2026-03-14 05:07:30.000000 UTC\n'
    '0xa3066fd0e2a0\tUDPv4\t0.0.0.0\t5353\t*\t0\t\t1100\tsvchost.exe\t2026-03-13 12:58:09.000000 UTC\n'
))
RESERVA['case-lab-p4/memory/windows.malfind.txt'] = ''.join((
    '# SYNTHETIC TRAINING DATA (case LAB-P4). Resembles Volatility 3 windows.malfind output for\n'
    '# fictional host WS-FIN-07. Hexdump bytes are illustrative only; nothing executable is here.\n'
    'PID\tProcess\tStart VPN\tEnd VPN\tTag\tProtection\tCommitCharge\tPrivateMemory\tFile output\tNotes\tHexdump\tDisasm\n'
    '\n'
    '7704\trundll32.exe\t0x1c0000\t0x1e0fff\tVadS\tPAGE_EXECUTE_READWRITE\t33\t1\tDisabled\tMZ header\n'
    '4d 5a 90 00 03 00 00 00 04 00 00 00 ff ff 00 00\tMZ..............\n'
    'b8 00 00 00 00 00 00 00 40 00 00 00 00 00 00 00\t........@.......\n'
    '00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00\t................\n'
    '00 00 00 00 00 00 00 00 00 00 00 00 f8 00 00 00\t................\n'
    '\n'
    '7316\tpowershell.exe\t0x7ff4a2c30000\t0x7ff4a2c3ffff\tVadS\tPAGE_EXECUTE_READWRITE\t2\t1\tDisabled\tN/A\n'
    '00 00 00 00 00 00 00 00 49 ba 60 1c 8d a2 f4 7f\t........I.`.....\n'
    '00 00 41 ff e3 cc cc cc cc cc cc cc cc cc cc cc\t..A.............\n'
    '49 ba 10 2d 8d a2 f4 7f 00 00 41 ff e3 cc cc cc\tI..-......A.....\n'
    'cc cc cc cc cc cc cc cc cc cc cc cc cc cc cc cc\t................\n'
    '\n'
    '6208\tmsedge.exe\t0x2b2c0040000\t0x2b2c007ffff\tVadS\tPAGE_EXECUTE_READWRITE\t64\t1\tDisabled\tN/A\n'
    '00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00\t................\n'
    '00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00\t................\n'
))

EVIDENCIA = {}           # o que você vai analisar de fato (nome curto -> texto)
ORIGEM = {}              # de onde veio cada arquivo: "rede" ou "reserva"

for nome, reserva in RESERVA.items():
    texto = None
    if not FORCAR_OFFLINE:
        try:
            # timeout curto: se a rede não responder em 8 s, seguimos com o plano B
            with urllib.request.urlopen(URL_BASE + nome, timeout=8) as resp:
                texto = resp.read().decode("utf-8")
            ORIGEM[nome] = "rede"
        except Exception as erro:            # sem internet, proxy, 404... qualquer falha
            print(f"⚠️  {nome.split('/')[-1]}: rede indisponível ({type(erro).__name__}), usando a cópia embutida")
    if texto is None:
        texto = reserva
        ORIGEM[nome] = "reserva"
    EVIDENCIA[nome.split("/")[-1]] = texto   # guardamos pelo nome curto
    h = hashlib.sha256(texto.encode("utf-8")).hexdigest()
    # Se o upstream mudou depois que este notebook foi gerado, os hashes divergem: avisamos.
    igual = "= cópia embutida" if texto == reserva else "≠ cópia embutida (o upstream mudou!)"
    print(f"{ORIGEM[nome]:8} {h[:16]}…  {nome.split('/')[-1]}  ({igual})")


## 1. O `awk` do passo 1, em Python

`$1 ~ /^0x/` (linha de dados), `$5 != "0.0.0.0"` e `$5 != "*"` (tem remoto). Depois, ordenar pelo dono.


In [ ]:
def ler_netscan(texto):
    linhas = [l for l in texto.splitlines() if l.startswith("0x")]     # $1 ~ /^0x/
    cab = ["Offset", "Proto", "LocalAddr", "LocalPort", "ForeignAddr", "ForeignPort", "State", "PID", "Owner", "Created"]
    return [dict(zip(cab, l.split("\t"))) for l in linhas]

def externas(ns):
    return sorted((r for r in ns if r["ForeignAddr"] not in ("0.0.0.0", "*")),
                  key=lambda r: (r["Owner"], r["PID"], r["Created"]))

NS = ler_netscan(EVIDENCIA["windows.netscan.txt"])
ESPERADOS = {"192.0.2.30": "e-mail da empresa", "192.0.2.31": "web da empresa"}

def mostrar(ns, esperados):
    for r in externas(ns):
        tag = esperados.get(r["ForeignAddr"], "❓ NÃO esperado")
        print(f"{r['Owner']} ({r['PID']}) {r['LocalAddr']}:{r['LocalPort']} -> {r['ForeignAddr']}:{r['ForeignPort']} "
              f"{r['State'] or '-'} {r['Created'][:19]}   [{tag}]")
    print(f"{len(externas(ns))} linhas")

mostrar(NS, ESPERADOS)


## 2. O que cada linha prova

Para cada linha, a função devolve **a frase que você pode escrever**. Repare no `SYN_SENT` e no `CLOSED`.


In [ ]:
def frase(r):
    base = f"{r['Owner']} (PID {r['PID']}) "
    if r["State"] == "SYN_SENT":
        return base + f"TENTOU conectar a {r['ForeignAddr']}:{r['ForeignPort']} (SYN enviado, sem resposta registrada)."
    if r["State"] == "CLOSED":
        return base + (f"manteve um endpoint com {r['ForeignAddr']}:{r['ForeignPort']}, criado {r['Created'][11:19]}Z, "
                       "já fechado antes da captura.")
    return base + (f"manteve uma conexão ESTABLISHED com {r['ForeignAddr']}:{r['ForeignPort']}, criada {r['Created'][11:19]}Z "
                   "(em algum momento antes da captura).")

for r in externas(NS):
    print("•", frase(r))


### O intervalo do `synchelper.exe` (passo 2)


In [ ]:
from datetime import datetime

def t(s):
    return datetime.strptime(s[:19], "%Y-%m-%d %H:%M:%S")

def intervalos(ns, pid, ip):
    tempos = sorted(t(r["Created"]) for r in ns if r["PID"] == pid and r["ForeignAddr"] == ip)
    return [(b - a).total_seconds() for a, b in zip(tempos, tempos[1:])]

g = intervalos(NS, "7488", "198.51.100.23")
print("Intervalos (s):", g)
print("Um intervalo só não é padrão: é uma PERGUNTA para o Dia 59 (dados de rede).")
print("\nPasso 3: 198.51.100.23 também é o destino dos ~48 MB do fs01 (firewall.log). Escreva: 'mesmo IP remoto'.")


## 3. O malfind, região por região

Primeiro interpretamos o arquivo (cabeçalho de cada região + linhas de *hexdump*). Depois aplicamos o fluxograma:

`MZ?` → sim: provável PE · não → o processo faz JIT? → os bytes parecem JIT (*thunks* `49 ba … 41 ff e3`, ou só zeros)? → provavelmente benigno · senão: extrair e analisar.


In [ ]:
def ler_malfind(texto):
    regioes, atual = [], None
    for l in texto.splitlines():
        if not l.strip() or l.startswith("#") or l.startswith("PID\t"):
            continue
        c = l.split("\t")
        if c[0].isdigit():                          # linha de cabeçalho da região
            atual = dict(PID=c[0], Process=c[1], Start=c[2], End=c[3], Protection=c[5],
                         Private=c[7], Notes=c[9], bytes=b"")
            regioes.append(atual)
        elif atual is not None:                     # linha de hexdump: "4d 5a 90 00 ...<TAB>MZ......"
            atual["bytes"] += bytes.fromhex(c[0])
    return regioes

PROCESSOS_JIT = {"powershell.exe", "msedge.exe", "chrome.exe", "firefox.exe"}   # .NET / JavaScript

def veredito(r, jit=PROCESSOS_JIT):
    b = r["bytes"]
    if b[:2] == b"MZ":
        return ("🔴 indicador forte", "MZ em memória privada RWX: provável PE escrito na memória. "
                "Próximo passo: --dump, hash, strings (Dia 61), dlllist.")
    if r["Process"] in jit:
        thunks = b.count(b"\x49\xba") >= 2 and b"\x41\xff\xe3" in b
        zeros = b.strip(b"\x00") == b""
        if thunks:
            return ("🔵 provavelmente benigno", "thunks repetidos mov r10/jmp r11 com cc: padrão de JIT .NET; fraco sozinho")
        if zeros:
            return ("🔵 benigno / provavelmente benigno", "só zeros num processo com JIT: alocação ainda não preenchida")
    return ("🟠 indeterminado", "não é MZ nem padrão de JIT conhecido: extrair, hash, strings")

MF = ler_malfind(EVIDENCIA["windows.malfind.txt"])
for r in MF:
    v, porque = veredito(r)
    print(f"{r['PID']:>5} {r['Process']:15} {r['Protection']} privada={r['Private']}  "
          f"primeiros: {r['bytes'][:4].hex(' ')}\n      {v}: {porque}")


## 4. O grafo processo-rede

Caixas = processos, elipses = IPs, losangos = regiões do malfind. `CLOSED` tracejado, `SYN_SENT` pontilhado, esperados em cinza.


In [ ]:
import shutil, subprocess
from pathlib import Path

def grafo(ns, mf, cadeia, esperados, nome, notas_ip=None):
    notas_ip = notas_ip or {}
    out = [f"digraph {nome} {{", "  rankdir=LR;"]
    vered = {r["PID"]: veredito(r)[0] for r in mf}
    pids = {r["PID"]: r["Owner"] for r in externas(ns)} | {r["PID"]: r["Process"] for r in mf}
    for pid, nomep in pids.items():
        cor = ', style=filled, fillcolor="#f4b6b6"' if vered.get(pid, "").startswith("🔴") or pid in cadeia else ""
        out.append(f'  p{pid} [shape=box, label="{nomep} ({pid})"{cor}];')
    for r in externas(ns):
        ip = f'ip{r["ForeignAddr"].replace(".", "_")}_{r["ForeignPort"]}'
        cinza = r["ForeignAddr"] in esperados
        extra = notas_ip.get(r["ForeignAddr"], "")
        out.append(f'  {ip} [shape=ellipse, label="{r["ForeignAddr"]}:{r["ForeignPort"]}{extra}"'
                   f'{", color=gray, fontcolor=gray" if cinza else ""}];')
        estilo = {"CLOSED": ", style=dashed", "SYN_SENT": ", style=dotted"}.get(r["State"], "")
        out.append(f'  p{r["PID"]} -> {ip} [label="{r["State"]} {r["Created"][11:19]}Z"{estilo}'
                   f'{", color=gray" if cinza else ""}];')
    for r in mf:
        rid = f'r{r["PID"]}'
        cor = "#f4b6b6" if vered[r["PID"]].startswith("🔴") else "#cfe3f6"
        marca = "MZ header" if r["bytes"][:2] == b"MZ" else ("JIT, likely benign" if "benigno" in vered[r["PID"]] else "?")
        out.append(f'  {rid} [shape=diamond, style=filled, fillcolor="{cor}", label="{r["Start"]} RWX\\n{marca}"];')
        out.append(f'  p{r["PID"]} -> {rid} [label="{r["Protection"]}"];')
    for pai, filho in cadeia.items():
        if pai in pids and filho in pids:
            out.append(f"  p{pai} -> p{filho} [color=gray];")
    out.append("}")
    return "\n".join(out)

CADEIA = {"7316": "7488", "7488": "7704"}       # parent -> child, cadeia suspeita do Dia 53
dot = grafo(NS, MF, CADEIA, ESPERADOS, "net", {"198.51.100.23": "\\nalso fs01 egress (firewall.log)"})
Path("ws-fin-07-process-network.dot").write_text(dot, encoding="utf-8")
print(dot)
if shutil.which("dot"):
    subprocess.run(["dot", "-Tpng", "ws-fin-07-process-network.dot", "-o", "ws-fin-07-process-network.png"], check=False)
    print("\n🖼️  ws-fin-07-process-network.png")

print("\nPasso 6: SYN_SENT = o svchost.exe 5124 enviou o SYN para 203.0.113.80:8443 e a conexão não se completou.",
      "\nNunca escreva 'conectado a'.")


## 5. ➕ O `SA-CX-07`


In [ ]:
# ------------------------------------------------------------
# RECEBER NETSCAN/MALFIND DO SA-CX-07 E O PROXY DO DIA 49
# Plano A: baixar do repositório original (GitHub raw), com urllib.
# Plano B: usar a cópia embutida logo abaixo (o arquivo inteiro,
#          copiado do upstream/), para o notebook rodar sem internet.
# ------------------------------------------------------------
import urllib.request      # baixa arquivos da web, sem instalar nada
import hashlib             # SHA-256: a impressão digital dos bytes

FORCAR_OFFLINE = False     # True = nem tenta a rede (útil em sala sem Wi-Fi)
URL_BASE = 'https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/main/caso-continuo/dados/'

RESERVA = {}             # caminho do arquivo -> conteúdo embutido (plano B)
RESERVA['dia-54/sa-cx-07.windows.netscan.txt'] = ''.join((
    '# DADOS SINTÉTICOS (Operação Isca). Texto escrito à mão no layout do Volatility 3 para o host\n'
    '# fictício SA-CX-07 (10.20.40.57). NÃO existe imagem de memória por trás. Separado por tabulação.\n'
    'Offset\tProto\tLocalAddr\tLocalPort\tForeignAddr\tForeignPort\tState\tPID\tOwner\tCreated\n'
    '0xb2052a10a010\tTCPv4\t0.0.0.0\t135\t0.0.0.0\t0\tLISTENING\t868\tsvchost.exe\t2026-10-05 10:50:44.000000 UTC\n'
    '0xb2052a10b8e0\tTCPv4\t0.0.0.0\t445\t0.0.0.0\t0\tLISTENING\t4\tSystem\t2026-10-05 10:50:46.000000 UTC\n'
    '0xb2052a2c0c20\tTCPv4\t0.0.0.0\t49664\t0.0.0.0\t0\tLISTENING\t716\tlsass.exe\t2026-10-05 10:50:44.000000 UTC\n'
    '0xb2052c7710a0\tTCPv4\t10.20.40.57\t50122\t10.20.5.30\t8443\tESTABLISHED\t6120\tcaixa-sa.exe\t2026-10-05 13:15:41.000000 UTC\n'
    '0xb2052c9a3b20\tTCPv4\t10.20.40.57\t51690\t192.0.2.24\t443\tESTABLISHED\t5216\tmsedge.exe\t2026-10-05 17:31:12.000000 UTC\n'
    '0xb2052b0e4410\tTCPv4\t10.20.40.57\t51031\t203.0.113.77\t443\tCLOSED\t5216\tmsedge.exe\t2026-10-05 11:21:05.000000 UTC\n'
    '0xb2052d1e98a0\tTCPv4\t10.20.40.57\t51702\t198.51.100.140\t443\tESTABLISHED\t7712\tmsa-atualizado\t2026-10-05 17:40:14.000000 UTC\n'
    '0xb2052d1ef010\tTCPv4\t10.20.40.57\t51697\t198.51.100.140\t443\tCLOSED\t7712\tmsa-atualizado\t2026-10-05 17:35:09.000000 UTC\n'
    '0xb2052d2a0a10\tUDPv4\t10.20.40.57\t58211\t*\t0\t\t7712\tmsa-atualizado\t2026-10-05 17:40:13.000000 UTC\n'
    '0xb2052a3e12a0\tUDPv4\t0.0.0.0\t5353\t*\t0\t\t1204\tsvchost.exe\t2026-10-05 10:50:49.000000 UTC\n'
))
RESERVA['dia-54/sa-cx-07.windows.malfind.txt'] = ''.join((
    '# DADOS SINTÉTICOS (Operação Isca). Texto escrito à mão no layout do Volatility 3 para o host\n'
    '# fictício SA-CX-07 (10.20.40.57). NÃO existe imagem de memória por trás. Bytes ilustrativos; nada executável aqui.\n'
    'PID\tProcess\tStart VPN\tEnd VPN\tTag\tProtection\tCommitCharge\tPrivateMemory\tFile output\tNotes\tHexdump\tDisasm\n'
    '\n'
    '7712\tmsa-atualizado\t0x2a10000\t0x2a3ffff\tVadS\tPAGE_EXECUTE_READWRITE\t48\t1\tDisabled\tMZ header\n'
    '4d 5a 90 00 03 00 00 00 04 00 00 00 ff ff 00 00\tMZ..............\n'
    'b8 00 00 00 00 00 00 00 40 00 00 00 00 00 00 00\t........@.......\n'
    '00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00\t................\n'
    '00 00 00 00 00 00 00 00 00 00 00 00 80 00 00 00\t................\n'
    '\n'
    '6120\tcaixa-sa.exe\t0x7ff8b1c20000\t0x7ff8b1c2ffff\tVadS\tPAGE_EXECUTE_READWRITE\t2\t1\tDisabled\tN/A\n'
    '00 00 00 00 00 00 00 00 49 ba 40 2c 1d b1 f8 7f\t........I.@,....\n'
    '00 00 41 ff e3 cc cc cc cc cc cc cc cc cc cc cc\t..A.............\n'
    '49 ba 90 3d 1d b1 f8 7f 00 00 41 ff e3 cc cc cc\tI..=......A.....\n'
    'cc cc cc cc cc cc cc cc cc cc cc cc cc cc cc cc\t................\n'
    '\n'
    '5216\tmsedge.exe\t0x1f3a0040000\t0x1f3a007ffff\tVadS\tPAGE_EXECUTE_READWRITE\t64\t1\tDisabled\tN/A\n'
    '00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00\t................\n'
    '00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00\t................\n'
))
RESERVA['dia-49/EVID-SA-001/proxy-sa.log'] = ''.join((
    'Oct  5 11:13:25 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT portal-noticias.example:443 200 out=3547 in=65569 dst=192.0.2.25\n'
    'Oct  5 11:18:47 proxy-sa squid[2211]: 10.20.40.55 atd.centro02 CONNECT portal-noticias.example:443 200 out=3123 in=73679 dst=192.0.2.11\n'
    'Oct  5 11:19:38 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 GET http://lnk.example/sa7 302 out=412 in=380 dst=198.51.100.201\n'
    'Oct  5 11:19:41 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT serraazul-pix.example:443 200 out=1840 in=48120 dst=203.0.113.77\n'
    'Oct  5 11:21:05 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT serraazul-pix.example:443 200 out=2311 in=5120 dst=203.0.113.77\n'
    'Oct  5 11:21:52 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT serraazul-pix.example:443 200 out=2290 in=4980 dst=203.0.113.77\n'
    'Oct  5 11:23:12 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT serraazul-pix.example:443 200 out=1622 in=412880 dst=203.0.113.77\n'
    'Oct  5 11:25:02 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1198 in=796 dst=198.51.100.140\n'
    'Oct  5 11:26:02 proxy-sa squid[2211]: 10.20.40.55 atd.centro02 CONNECT busca.example:443 200 out=2799 in=72449 dst=192.0.2.27\n'
    'Oct  5 11:29:54 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1227 in=630 dst=198.51.100.140\n'
    'Oct  5 11:34:49 proxy-sa squid[2211]: 10.20.40.51 cx.centro05 CONNECT portal-noticias.example:443 200 out=3094 in=55049 dst=192.0.2.35\n'
    'Oct  5 11:34:56 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1543 in=741 dst=198.51.100.140\n'
    'Oct  5 11:39:41 proxy-sa squid[2211]: 10.20.40.53 cx.centro06 CONNECT intranet-cdn.example:443 200 out=2473 in=4780 dst=192.0.2.16\n'
    'Oct  5 11:39:47 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1279 in=771 dst=198.51.100.140\n'
    'Oct  5 11:43:55 proxy-sa squid[2211]: 10.20.40.51 cx.centro05 CONNECT mapas.example:443 200 out=2981 in=8490 dst=192.0.2.16\n'
    'Oct  5 11:44:51 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1303 in=722 dst=198.51.100.140\n'
    'Oct  5 11:48:08 proxy-sa squid[2211]: 10.20.40.53 cx.centro06 CONNECT mapas.example:443 200 out=1085 in=61379 dst=192.0.2.23\n'
    'Oct  5 11:49:50 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1525 in=806 dst=198.51.100.140\n'
    'Oct  5 11:54:59 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1597 in=721 dst=198.51.100.140\n'
    'Oct  5 11:57:59 proxy-sa squid[2211]: 10.20.40.52 ger.centro01 CONNECT busca.example:443 200 out=2448 in=28809 dst=192.0.2.12\n'
    'Oct  5 11:59:52 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1477 in=826 dst=198.51.100.140\n'
    'Oct  5 12:04:45 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1323 in=899 dst=198.51.100.140\n'
    'Oct  5 12:07:10 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=3145728 in=1022 dst=198.51.100.140\n'
    'Oct  5 12:09:42 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1106 in=629 dst=198.51.100.140\n'
    'Oct  5 12:14:35 proxy-sa squid[2211]: 10.20.40.52 ger.centro01 CONNECT busca.example:443 200 out=1480 in=5140 dst=192.0.2.22\n'
    'Oct  5 12:14:49 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1277 in=700 dst=198.51.100.140\n'
    'Oct  5 12:17:49 proxy-sa squid[2211]: 10.20.40.52 ger.centro01 CONNECT intranet-cdn.example:443 200 out=2959 in=37419 dst=192.0.2.16\n'
    'Oct  5 12:19:58 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1350 in=709 dst=198.51.100.140\n'
    'Oct  5 12:24:59 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1585 in=627 dst=198.51.100.140\n'
    'Oct  5 12:26:36 proxy-sa squid[2211]: 10.20.40.52 ger.centro01 CONNECT fornecedor-cartoes.example:443 200 out=1070 in=32479 dst=192.0.2.24\n'
    'Oct  5 12:30:03 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1492 in=730 dst=198.51.100.140\n'
    'Oct  5 12:30:32 proxy-sa squid[2211]: 10.20.40.51 cx.centro05 CONNECT portal-noticias.example:443 200 out=2525 in=5542 dst=192.0.2.30\n'
    'Oct  5 12:35:05 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1101 in=854 dst=198.51.100.140\n'
    'Oct  5 12:36:01 proxy-sa squid[2211]: 10.20.40.53 cx.centro06 CONNECT mapas.example:443 200 out=2206 in=34292 dst=192.0.2.18\n'
))

SA = {}           # o que você vai analisar de fato (nome curto -> texto)
ORIGEM = {}              # de onde veio cada arquivo: "rede" ou "reserva"

for nome, reserva in RESERVA.items():
    texto = None
    if not FORCAR_OFFLINE:
        try:
            # timeout curto: se a rede não responder em 8 s, seguimos com o plano B
            with urllib.request.urlopen(URL_BASE + nome, timeout=8) as resp:
                texto = resp.read().decode("utf-8")
            ORIGEM[nome] = "rede"
        except Exception as erro:            # sem internet, proxy, 404... qualquer falha
            print(f"⚠️  {nome.split('/')[-1]}: rede indisponível ({type(erro).__name__}), usando a cópia embutida")
    if texto is None:
        texto = reserva
        ORIGEM[nome] = "reserva"
    SA[nome.split("/")[-1]] = texto   # guardamos pelo nome curto
    h = hashlib.sha256(texto.encode("utf-8")).hexdigest()
    # Se o upstream mudou depois que este notebook foi gerado, os hashes divergem: avisamos.
    igual = "= cópia embutida" if texto == reserva else "≠ cópia embutida (o upstream mudou!)"
    print(f"{ORIGEM[nome]:8} {h[:16]}…  {nome.split('/')[-1]}  ({igual})")


In [ ]:
NS_SA = ler_netscan(SA["sa-cx-07.windows.netscan.txt"])
ESPERADOS_SA = {"10.20.5.30": "servidor do sistema de caixa", "192.0.2.24": "portal de notícias (visto em outros caixas)"}
mostrar(NS_SA, ESPERADOS_SA)
print()
for r in externas(NS_SA):
    if r["ForeignAddr"] not in ESPERADOS_SA:
        print("•", frase(r))
print("\n203.0.113.77 = IP do site falso serraazul-pix.example (fatos fixos do caso).")


### O teste do padrão: memória + proxy

Duas linhas de memória dão **um** intervalo. O proxy (UTC, Dia 49) dá uma série. Vamos medir a regularidade com `statistics`.


In [ ]:
from statistics import mean, pstdev

g_mem = intervalos(NS_SA, "7712", "198.51.100.140")
print("Memória (7712 -> 198.51.100.140):", g_mem, "s  -> uma pergunta")

linhas = [l for l in SA["proxy-sa.log"].splitlines() if "10.20.40.57" in l and "atualiza-modulo.example" in l]
horas = [datetime.strptime(l[7:15], "%H:%M:%S") for l in linhas]
gaps = [(b - a).total_seconds() for a, b in zip(horas, horas[1:])]
print(f"\nProxy: {len(linhas)} conexões de {linhas[0][7:15]} a {linhas[-1][7:15]} UTC")
print("Intervalos:", [int(x) for x in gaps])

# A linha de 3 MiB está fora do ritmo: separamos as "fora de ritmo" (menos de 200 s desde a anterior)
ritmo = [x for x in gaps if x >= 200]
print(f"Sem as quebras de ritmo: média {mean(ritmo):.0f} s, coeficiente de variação {pstdev(ritmo) / mean(ritmo):.1%}")

grande = [l for l in linhas if int(l.split("out=")[1].split()[0]) > 100_000]
for l in grande:
    print("\nFora do ritmo:", l)
print("O que dá para afirmar: ~3 MiB saíram para 198.51.100.140 num túnel TLS. Conteúdo: NÃO observado.")


In [ ]:
MF_SA = ler_malfind(SA["sa-cx-07.windows.malfind.txt"])
JIT_SA = PROCESSOS_JIT | {"caixa-sa.exe"}         # o sistema de caixa é .NET (fato do caso, informado pela TI)
for r in MF_SA:
    v, porque = veredito(r, JIT_SA)
    print(f"{r['PID']:>5} {r['Process']:15} primeiros: {r['bytes'][:4].hex(' ')}\n      {v}: {porque}")

dot_sa = grafo(NS_SA, MF_SA, {}, ESPERADOS_SA, "net_sa",
               {"198.51.100.140": "\\natualiza-modulo.example (proxy)", "203.0.113.77": "\\nserraazul-pix.example"})
Path("sa-cx-07-process-network.dot").write_text(dot_sa, encoding="utf-8")
print("\n💾 sa-cx-07-process-network.dot")


## 6. 🎮 Quiz: o que esta linha prova?

Para cada afirmação, responda **S** se a evidência citada a sustenta sozinha, **N** se não.


In [ ]:
MODO_JOGO = True
AFIRMACOES = [
    ("netscan: svchost 5124 SYN_SENT para 203.0.113.80:8443 -> '5124 estava conectado a 203.0.113.80'", "N",
     "SYN_SENT é tentativa não concluída."),
    ("netscan: synchelper 7488 CLOSED 05:02:30 -> '7488 manteve um endpoint com 198.51.100.23 antes da captura'", "S",
     "É exatamente o que uma linha do netscan prova."),
    ("198.51.100.23 aparece no WS-FIN-07 e no egress do fs01 -> 'o mesmo atacante agiu nos dois hosts'", "N",
     "Mesmo IP remoto. Atacante é outra afirmação."),
    ("malfind: zeros em RWX no msedge -> 'injeção no navegador'", "N", "JIT não preenchido, quase sempre."),
    ("malfind: MZ em RWX privada no rundll32 sem DLL -> 'indicador forte de injeção'", "S",
     "É o caso clássico; confirme com --dump e dlllist."),
    ("➕ proxy: 3.145.728 bytes para 198.51.100.140 -> 'vazaram dados de cooperados'", "N", "Conteúdo não observado."),
    ("➕ proxy: 16 conexões a ~300 s com CV baixo + netscan com 2 conexões -> 'conexões a intervalo regular de ~300 s'",
     "S", "Duas fontes, cada uma dizendo o que mostra."),
]
acertos = 0
for i, (af, certa, porque) in enumerate(AFIRMACOES, 1):
    print(f"\n[{i}] {af}")
    if MODO_JOGO:
        r = ""
        while r not in ("S", "N"):
            r = input("   Sustenta? (S/N): ").strip().upper()[:1]
        acertos += r == certa
        print("   ✅" if r == certa else "   ❌", end=" ")
    print(f"{certa}. {porque}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(AFIRMACOES)}")


In [ ]:
try:
    from google.colab import files
    for f in ("ws-fin-07-process-network.dot", "sa-cx-07-process-network.dot"):
        files.download(f)
except ImportError:
    print("(Fora do Colab: os .dot ficaram na pasta atual)")


## ✅ Antes de fechar o Dia 54

- [ ] grafo com todas as linhas não-*listening* e todas as regiões do malfind, rotuladas
- [ ] tabela do malfind: Edge benigno, PowerShell com motivo, rundll32 com próximo passo
- [ ] nota do `SYN_SENT` sem "conectado a"
- [ ] ➕ `SA-CX-07`: 203.0.113.77 identificado; *beacon* com duas fontes; "conteúdo não observado" nos 3 MiB

**Amanhã (Dia 55):** logs de autenticação. Quem logou, de onde, como e quando, e o mapa de calor que faz a hora estranha saltar aos olhos.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>
